# Kafka Service - Produce All Data Sources

## 🌐 Service UIs (Click These!)
- **HDFS UI**: http://localhost:9870 | **File Browser**: http://localhost:9870/explorer.html#/lakehouse
- **Trino (Query)**: http://localhost:8083
- **Airflow (Orchestration)**: http://localhost:8090 (user: airflow, password: airflow)
- **Spark Master** (Docker): http://localhost:8082 | **Spark UI** (local session): http://localhost:4040
- **Grafana (Monitoring)**: http://localhost:3000 (user: admin, password: admin)
- **Prometheus (Metrics)**: http://localhost:9090
- **Schema Registry**: http://localhost:8081
- **PostgreSQL**: localhost:5432 (user: airflow, password: airflow, db: gold_layer)
- **Kafka Brokers**: localhost:19092, localhost:19093, localhost:19094
- **Zookeeper**: localhost:2181

**📖 For detailed service info, see** [START_HERE.md](../START_HERE.md)

This notebook produces all CSV data sources to Kafka topics:
- `bookings_raw` - Booking transactions
- `booking_events_raw` - Booking status change events
- `hotels_raw` - Hotel reference data

In [1]:

from src.kafka.producer import KafkaProducer
import pandas as pd
import json

## 1. Load All CSV Files

In [2]:
# Load bookings
bookings_df = pd.read_csv('../data/bookings_raw.csv')
print(f"✅ Loaded {len(bookings_df)} bookings")
print(bookings_df.head())

# Load booking events
events_df = pd.read_csv('../data/booking_events_raw.csv')
print(f"\n✅ Loaded {len(events_df)} booking events")
print(events_df.head())

# Load hotels
hotels_df = pd.read_csv('../data/hotels_raw.csv')
print(f"\n✅ Loaded {len(hotels_df)} hotels")
print(hotels_df.head())

✅ Loaded 10 bookings
  booking_id user_id hotel_id     status  price           created_at  \
0         b1      u1       h1    created  120.0  2025-01-01T10:05:00   
1         b1      u1       h1  confirmed  120.0  2025-01-01T10:05:00   
2         b2      u2       h2    created  200.0  2025-01-01T11:00:00   
3         b2      u2       h2  cancelled  200.0  2025-01-02T09:00:00   
4         b3      u3       h1    created  150.0  2025-01-02T14:20:00   

            updated_at  
0  2025-01-01T10:05:00  
1  2025-01-01T10:10:00  
2  2025-01-01T11:00:00  
3  2025-01-02T09:05:00  
4  2025-01-02T14:20:00  

✅ Loaded 11 booking events
  booking_id event_type             event_ts
0         b1    created  2025-01-01T10:05:00
1         b1  confirmed  2025-01-01T10:10:00
2         b2    created  2025-01-01T11:00:00
3         b2  cancelled  2025-01-02T09:05:00
4         b3    created  2025-01-02T14:20:00

✅ Loaded 3 hotels
  hotel_id     city  star_rating
0       h1   Tehran            4
1       h2   

In [3]:
# Check Kafka is reachable before producing (avoids confusing "Connection refused" later)
import os
import socket

def _parse_bootstrap(s):
    """Parse bootstrap string; handles 'a,b,c' or \"['a','b','c']\" (env list repr)."""
    s = (s or "").strip()
    if s.startswith("["):
        s = s[1:].rstrip("]").strip()
    return [b.strip().strip("'\"") for b in s.split(",") if b.strip()]

KAFKA_BOOTSTRAP = os.environ.get("KAFKA_BOOTSTRAP_SERVERS", "localhost:19092,localhost:19093,localhost:19094")
bootstrap_list = _parse_bootstrap(KAFKA_BOOTSTRAP) or ["localhost:19092"]
first_broker = bootstrap_list[0]
host, _, port_str = first_broker.rpartition(":")
port = int(port_str) if port_str else 19092
try:
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(2)
    s.connect((host or "localhost", port))
    s.close()
    print(f"✅ Kafka reachable at {first_broker}")
except (socket.error, OSError, ValueError) as e:
    raise RuntimeError(
        "Kafka is not reachable. Start Docker: run 'make docker-up' from the project root\n"
        "(snapptrip-data-platform/ or its parent folder). Then re-run this cell.\n"
        f"Kafka should listen on {first_broker}."
    ) from e

✅ Kafka reachable at kafka-1:9092


## 2. Initialize Kafka Producer

**Prerequisite:** Start Docker from the project root so Kafka is running: `make docker-up`. Kafka will listen on `localhost:19092`, `19093`, `19094`. If you see "Connection refused", run that command first.

In [4]:
# Use same bootstrap as the rest of the platform (override with KAFKA_BOOTSTRAP_SERVERS env)
from src.kafka.producer import KafkaProducer
import os
import socket

def _parse_bootstrap(s):
    """Parse bootstrap string; handles 'a,b,c' or \"['a','b','c']\" (env list repr)."""
    s = (s or "").strip()
    if s.startswith("["):
        s = s[1:].rstrip("]").strip()
    return [b.strip().strip("'\"") for b in s.split(",") if b.strip()]

KAFKA_BOOTSTRAP = os.environ.get("KAFKA_BOOTSTRAP_SERVERS", "localhost:19092,localhost:19093,localhost:19094")
bootstrap_list = _parse_bootstrap(KAFKA_BOOTSTRAP) or ["localhost:19092"]

# Fail fast with a clear message if Kafka is not running
first = bootstrap_list[0]
host, _, port_str = first.rpartition(":")
port = int(port_str) if port_str else 19092
try:
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(3)
    s.connect((host or "localhost", port))
    s.close()
except (socket.error, OSError, ValueError) as e:
    raise RuntimeError(
        "Kafka is not reachable. From the project root run:\n  make docker-up\n"
        f"Then ensure Kafka is listening on {first}."
    ) from e

# Initialize producer with Kafka bootstrap servers
producer = KafkaProducer(bootstrap_servers=bootstrap_list)
print(f"✅ Kafka producer initialized ({KAFKA_BOOTSTRAP})")

✅ Kafka producer initialized (kafka-1:9092,kafka-2:9092,kafka-3:9092)


## 3. Produce Bookings to Kafka

In [5]:
# Produce all bookings to bookings_raw topic
if 'producer' not in globals():
    raise RuntimeError(
        "Run the 'Initialize Kafka Producer' cell (Section 2) first. "
        "If that cell failed, start Docker with 'make docker-up', then run it again."
    )
print("Producing bookings to 'bookings_raw' topic...")
for _, row in bookings_df.iterrows():
    message = {
        'booking_id': str(row['booking_id']),
        'user_id': str(row['user_id']),
        'hotel_id': str(row['hotel_id']),
        'status': str(row['status']),
        'price': float(row['price']),
        'created_at': str(row['created_at']),
        'updated_at': str(row['updated_at'])
    }
    producer.produce('bookings_raw', key=message['booking_id'], value=message)

producer.flush()
print(f"✅ Produced {len(bookings_df)} bookings to 'bookings_raw' topic")

Producing bookings to 'bookings_raw' topic...
✅ Produced 10 bookings to 'bookings_raw' topic


## 4. Produce Booking Events to Kafka

In [6]:
# Produce all booking events to booking_events_raw topic
if 'producer' not in globals():
    raise RuntimeError("Run the 'Initialize Kafka Producer' cell (Section 2) first. If it failed, start Docker: make docker-up, then run it again.")
print("Producing booking events to 'booking_events_raw' topic...")
for _, row in events_df.iterrows():
    message = {
        'booking_id': str(row['booking_id']),
        'event_type': str(row['event_type']),
        'event_ts': str(row['event_ts'])
    }
    producer.produce('booking_events_raw', key=message['booking_id'], value=message)

producer.flush()
print(f"✅ Produced {len(events_df)} events to 'booking_events_raw' topic")

Producing booking events to 'booking_events_raw' topic...
✅ Produced 11 events to 'booking_events_raw' topic


## 5. Produce Hotels to Kafka

In [7]:
# Produce all hotels to hotels_raw topic
if 'producer' not in globals():
    raise RuntimeError("Run the 'Initialize Kafka Producer' cell (Section 2) first. If it failed, start Docker: make docker-up, then run it again.")
print("Producing hotels to 'hotels_raw' topic...")
for _, row in hotels_df.iterrows():
    message = {
        'hotel_id': str(row['hotel_id']),
        'city': str(row['city']),
        'star_rating': int(row['star_rating'])
    }
    producer.produce('hotels_raw', key=message['hotel_id'], value=message)

producer.flush()
print(f"✅ Produced {len(hotels_df)} hotels to 'hotels_raw' topic")

Producing hotels to 'hotels_raw' topic...
✅ Produced 3 hotels to 'hotels_raw' topic


## 6. Verify Topics

In [8]:
import os
from confluent_kafka.admin import AdminClient
from confluent_kafka import Consumer, TopicPartition

bootstrap = os.environ.get("KAFKA_BOOTSTRAP_SERVERS", "localhost:19092,localhost:19093,localhost:19094")
conf = {"bootstrap.servers": bootstrap}

# List Kafka topics (works in Docker and on host)
admin = AdminClient(conf)
metadata = admin.list_topics(timeout=10)
topics = sorted(metadata.topics.keys())
print("📋 Kafka Topics:")
for t in topics:
    print(t)

# Get high-water mark (approx message count) per partition for our topics
for topic in ['bookings_raw', 'booking_events_raw', 'hotels_raw']:
    if topic not in metadata.topics:
        print(f"\n📊 Topic '{topic}': (not found)")
        continue
    c = Consumer({**conf, "group.id": "verify-list"})
    parts = metadata.topics[topic].partitions
    out = []
    for pid in sorted(parts.keys()):
        lo, hi = c.get_watermark_offsets(TopicPartition(topic, pid), timeout=5)
        out.append(f"{topic}:{pid}:{hi}")
    c.close()
    print(f"\n📊 Topic '{topic}': " + " ".join(out))

📋 Kafka Topics:
__consumer_offsets
_schemas
booking_events_raw
bookings_raw
hotels_raw

📊 Topic 'bookings_raw': bookings_raw:0:0 bookings_raw:1:120 bookings_raw:2:30

📊 Topic 'booking_events_raw': booking_events_raw:0:0 booking_events_raw:1:120 booking_events_raw:2:45

📊 Topic 'hotels_raw': hotels_raw:0:45


## Summary

### Data Produced:
- ✅ **bookings_raw**: 10 booking records
- ✅ **booking_events_raw**: 11 event records
- ✅ **hotels_raw**: 3 hotel records

### Next Step:
Run **02_bronze_layer.ipynb** to ingest these Kafka topics into HDFS Iceberg tables using DBT.